# Goldstein-price function

**Contributed by**: Benoît Legat

In this example, we consider the minimization of the [Goldstein-price function](https://en.wikipedia.org/wiki/Test_functions_for_optimization).

In [1]:
using SumOfSquares
using DynamicPolynomials

Create *symbolic* variables (not JuMP *decision* variables)

In [2]:
@polyvar x[1:2]

(DynamicPolynomials.Variable{DynamicPolynomials.Commutative{DynamicPolynomials.CreationOrder}, MultivariatePolynomials.Graded{MultivariatePolynomials.LexOrder}}[x₁, x₂],)

To use Sum-of-Squares Programming, we first need to pick an SDP solver,
see [here](https://jump.dev/JuMP.jl/stable/installation/#Supported-solvers) for a list of the available choices.

In [3]:
import Clarabel
using Dualization
model = SOSModel(dual_optimizer(Clarabel.Optimizer))

A JuMP Model
├ solver: Dual model with Clarabel attached
├ objective_sense: FEASIBILITY_SENSE
├ num_variables: 0
├ num_constraints: 0
└ Names registered in the model: none

Create a JuMP decision variable for the lower bound

In [4]:
@variable(model, γ)

γ

`f(x)` is the Goldstein-Price function

In [5]:
f1 = x[1] + x[2] + 1
f2 = 19 - 14*x[1] + 3*x[1]^2 - 14*x[2] + 6*x[1]*x[2] + 3*x[2]^2
f3 = 2*x[1] - 3*x[2]
f4 = 18 - 32*x[1] + 12*x[1]^2 + 48*x[2] - 36*x[1]*x[2] + 27*x[2]^2
f = (1 + f1^2*f2) * (30 + f3^2*f4)

600 + 720x₂ + 720x₁ + 3060x₂² - 4680x₁x₂ + 1260x₁² + 12288x₂³ - 19296x₁x₂² + 7344x₁²x₂ - 1072x₁³ + 14346x₂⁴ - 23616x₁x₂³ + 7776x₁²x₂² + 5784x₁³x₂ - 2454x₁⁴ + 1944x₂⁵ - 11880x₁x₂⁴ + 5040x₁²x₂³ + 9840x₁³x₂² - 7680x₁⁴x₂ + 1344x₁⁵ - 4428x₂⁶ - 1188x₁x₂⁵ + 8730x₁²x₂⁴ + 1240x₁³x₂³ - 5370x₁⁴x₂² - 168x₁⁵x₂ + 952x₁⁶ - 648x₂⁷ + 1944x₁x₂⁶ + 3672x₁²x₂⁵ - 3480x₁³x₂⁴ - 4080x₁⁴x₂³ + 2592x₁⁵x₂² + 1344x₁⁶x₂ - 768x₁⁷ + 729x₂⁸ + 972x₁x₂⁷ - 1458x₁²x₂⁶ - 1836x₁³x₂⁵ + 1305x₁⁴x₂⁴ + 1224x₁⁵x₂³ - 648x₁⁶x₂² - 288x₁⁷x₂ + 144x₁⁸

Constraints `f(x) - γ` to be a sum of squares

In [6]:
con_ref = @constraint(model, f >= γ)
@objective(model, Max, γ)
optimize!(model)

-------------------------------------------------------------
           Clarabel.jl v0.11.1  -  Clever Acronym              

                   (c) Paul Goulart                          
                University of Oxford, 2022                   
-------------------------------------------------------------

problem:
  variables     = 45
  constraints   = 121
  nnz(P)        = 0
  nnz(A)        = 121
  cones (total) = 2
    : Zero        = 1,  numel = 1
    : PSDTriangle = 1,  numel = 120

settings:
  linear algebra: direct / qdldl, precision: 64 bit (1 thread)
  max iter = 200, time limit = Inf,  max step = 0.990
  tol_feas = 1.0e-08, tol_gap_abs = 1.0e-08, tol_gap_rel = 1.0e-08,
  static reg : on, ϵ1 = 1.0e-08, ϵ2 = 4.9e-32
  dynamic reg: on, ϵ = 1.0e-13, δ = 2.0e-07
  iter refine: on, reltol = 1.0e-13, abstol = 1.0e-12, 
               max iter = 10, stop ratio = 5.0
  equilibrate: on, min_scale = 1.0e-04, max_scale = 1.0e+04
               max iter = 10

iter    pcost        dc

The lower bound found is 3

In [7]:
solution_summary(model)

solution_summary(; result = 1, verbose = false)
├ solver_name          : Dual model with Clarabel attached
├ Termination
│ ├ termination_status : ALMOST_OPTIMAL
│ ├ result_count       : 1
│ └ raw_status         : ALMOST_SOLVED
├ Solution (result = 1)
│ ├ primal_status        : NEARLY_FEASIBLE_POINT
│ ├ dual_status          : NEARLY_FEASIBLE_POINT
│ ├ objective_value      : 3.00001e+00
│ └ dual_objective_value : 3.00001e+00
└ Work counters
  ├ solve_time (sec)   : 1.30664e-02
  └ barrier_iterations : 18

The moment matrix is as follows, we can already see the global minimizer
`[0, -1]` from the entries `(2, 1)` and `(3, 1)`.
This heuristic way to obtain solutions to the polynomial optimization problem
is suggested in [Laurent2008; (6.15)](@cite).

In [8]:
ν = moment_matrix(con_ref)

MomentMatrix with row/column basis:
 SubBasis{Monomial}([1, x[2], x[1], x[2]^2, x[1]*x[2], x[1]^2, x[2]^3, x[1]*x[2]^2, x[1]^2*x[2], x[1]^3, x[2]^4, x[1]*x[2]^3, x[1]^2*x[2]^2, x[1]^3*x[2], x[1]^4])
And entries in a 15×15 SymMatrix{Float64}:
  1.000000000335982      -0.9999977427173574     …   4.903715057880436e-7
 -0.9999977427173574      0.9999955980865209         1.9900383106331322e-7
 -3.088058767589677e-6    3.088077236294944e-6       8.998097931178191e-7
  0.9999955933596235     -0.9999934452115975         7.846020535404317e-5
  3.085022671881432e-6   -3.045329150981973e-6       8.48760707568095e-5
  1.7929927903588537e-7   1.5196839214241395e-8  …   0.0001617807237762807
 -0.9999934625103103      0.9999913131718998         2.0632619525001065e-5
 -3.051490243746433e-6    3.075636177993065e-6       9.715778352658986e-5
  1.5019865880447528e-8   3.463468502314616e-8       0.00027184145540736
  1.3189364645484088e-7   1.1453791831831483e-7      0.0005401302465896751
  0.999991311906

Many entries of the matrix actually have the same moment.
We can obtain the following list of these moments without duplicates
(ignoring when difference of entries representing the same moments is below `1e-5`)

In [9]:
μ = moment_vector(ν, atol = 1e-5)

MomentVector{Float64, StarAlgebras.SubBasis{MultivariateBases.Polynomial{Monomial, Vector{DynamicPolynomials.Variable{DynamicPolynomials.Commutative{DynamicPolynomials.CreationOrder}, MultivariatePolynomials.Graded{MultivariatePolynomials.LexOrder}}}, Vector{Int64}}, Int64, Vector{Int64}, StarAlgebras.MappedBasis{MultivariateBases.Polynomial{Monomial, Vector{DynamicPolynomials.Variable{DynamicPolynomials.Commutative{DynamicPolynomials.CreationOrder}, MultivariatePolynomials.Graded{MultivariatePolynomials.LexOrder}}}, Vector{Int64}}, Vector{Int64}, MultivariatePolynomials.ExponentsIterator{MultivariatePolynomials.Graded{MultivariatePolynomials.LexOrder}, Nothing, Vector{Int64}}, MultivariateBases.Variables{Monomial, Vector{DynamicPolynomials.Variable{DynamicPolynomials.Commutative{DynamicPolynomials.CreationOrder}, MultivariatePolynomials.Graded{MultivariatePolynomials.LexOrder}}}}, typeof(MultivariatePolynomials.exponents)}, Vector{Vector{Int64}}}, Vector{Float64}}([1.000000000335982, 

The truncated moment matrix can then be obtained as follows

In [10]:
ν_truncated = moment_matrix(μ, monomials(x, 0:3))

MomentMatrix with row/column basis:
 SubBasis{Monomial}([1, x[2], x[1], x[2]^2, x[1]*x[2], x[1]^2, x[2]^3, x[1]*x[2]^2, x[1]^2*x[2], x[1]^3])
And entries in a 10×10 SymMatrix{Float64}:
  1.000000000335982      -0.9999977427173574     …  1.3189364645484088e-7
 -0.9999977427173574      0.9999955933596235        1.1394089943398841e-7
 -3.088058767589677e-6    3.085022671881432e-6      4.903715057880436e-7
  0.9999955933596235     -0.9999934625103103        4.2408593161529036e-8
  3.085022671881432e-6   -3.051490243746433e-6      1.9900383106331322e-7
  1.7929927903588537e-7   1.5019865880447528e-8  …  8.998097931178191e-7
 -0.9999934625103103      0.9999913119065144        3.0222741401442117e-5
 -3.051490243746433e-6    3.0749418648884353e-6     7.846020535404317e-5
  1.5019865880447528e-8   5.559315770644871e-8      8.48760707568095e-5
  1.3189364645484088e-7   1.1394089943398841e-7     0.0001617807237762807

Let's check if the flatness property is satisfied.
The rank of `ν_truncated` seems to be 1:

In [11]:
using LinearAlgebra
LinearAlgebra.svdvals(Matrix(ν_truncated.Q))
LinearAlgebra.rank(Matrix(ν_truncated.Q), rtol = 1e-3)
svdvals(Matrix(ν_truncated.Q))

10-element Vector{Float64}:
 3.999980407625857
 0.00025486383102845794
 4.676073113680599e-5
 9.573117668854677e-7
 1.9353834784388287e-7
 6.151094751817452e-8
 2.8228027712475767e-8
 2.811440587004269e-8
 1.921809381816059e-8
 8.31818997748015e-9

The rank of `ν` is clearly higher than 1, closer to 3:

In [12]:
svdvals(Matrix(ν.Q))

15-element Vector{Float64}:
 5.091341645761701
 1.7415266656871398
 1.1024729277799594
 0.0006557177535281969
 0.00017622074662644233
 4.851350932219942e-5
 4.159476511365617e-5
 1.1373230937269403e-6
 2.1099011227927422e-7
 5.4430845714451585e-8
 1.7473675954362494e-8
 8.877850565915509e-9
 1.7791919596183895e-9
 8.180259879687036e-10
 6.06794598580913e-11

Even if the flatness property is not satisfied, we can
still try extracting the minimizer with a low rank decomposition of rank 3.
We find the optimal solution again doing so:

In [13]:
atomic_measure(ν, FixedRank(3))

Atomic measure on the variables x[1], x[2] with 1 atoms:
 at [-3.0489419839814974e-6, -0.9999978184906222] with weight 1.0209805976101867

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*